# NeuralAlpha Model Training Runbook (Interview)

This notebook trains and persists XGBoost + LSTM artifacts for the backend runtime.
It follows the exact execution order requested by the backend model-loader contract.

Expected outputs:
- ml_pipeline/models/xgb.joblib
- ml_pipeline/models/lstm_close.keras
- ml_pipeline/models/close_scaler.pkl

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

project_root = Path.cwd()
if project_root.name.lower() == 'notebooks':
    project_root = project_root.parent.parent

print('Project root:', project_root)
print('Python:', sys.executable)


## 1) Install required runtime deps
Equivalent command: `pip install yfinance vaderSentiment --quiet`

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'yfinance', 'vaderSentiment', '--quiet'], check=True)
print('Dependency install complete')

## 2) Prepare data
Equivalent command: `python ml_pipeline/training/prepare_data.py`

In [ ]:
subprocess.run([sys.executable, str(project_root / 'ml_pipeline' / 'training' / 'prepare_data.py')], check=True)

## 3) Train XGBoost classifier
Equivalent command: `python ml_pipeline/training/train_xgboost.py`

In [ ]:
subprocess.run([sys.executable, str(project_root / 'ml_pipeline' / 'training' / 'train_xgboost.py')], check=True)

## 4) Train LSTM forecaster
Equivalent command: `python ml_pipeline/training/train_lstm.py`

In [ ]:
subprocess.run([sys.executable, str(project_root / 'ml_pipeline' / 'training' / 'train_lstm.py')], check=True)

## 5) Run backend remediation check
Equivalent command: `python backend/scripts/auto_fix_system.py`

In [ ]:
subprocess.run([sys.executable, str(project_root / 'backend' / 'scripts' / 'auto_fix_system.py')], check=True)

## 6) Verify required artifacts

In [ ]:
artifacts = [
    project_root / 'ml_pipeline' / 'models' / 'xgb.joblib',
    project_root / 'ml_pipeline' / 'models' / 'lstm_close.keras',
    project_root / 'ml_pipeline' / 'models' / 'close_scaler.pkl',
]
for path in artifacts:
    print(path, '=>', 'OK' if path.exists() else 'MISSING')

missing = [str(p) for p in artifacts if not p.exists()]
if missing:
    raise RuntimeError(f'Missing artifacts: {missing}')

print('All required artifacts exist.')

## 7) Enable strict real-model mode
After training, set in `.env`:

`ENFORCE_REAL_MODELS=true`